# Study 09: Ablation Study\n**Goal:** Systematically vary hyperparameters to understand their impact on segmentation quality.

## 1. Setup

In [ ]:
import sys, json, subprocess, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
print('Setup complete')

## 2. Ablation Configurations

In [ ]:
OUT = Path.cwd().parent / 'results' / 'ablation'
OUT.mkdir(parents=True, exist_ok=True)

ABLATIONS = {
    'baseline':     ([], 'Pretrained MobileNetV2, FocalLoss, lr=1e-3, bs=8'),
    'no_pretrain':  (['--no-pretrained'], 'Scratch init (no ImageNet)'),
    'lr_1e-4':      (['--lr', '1e-4'], 'Lower learning rate'),
    'lr_1e-2':      (['--lr', '1e-2'], 'Higher learning rate'),
    'bs_4':         (['--batch-size', '4'], 'Smaller batch size'),
    'bs_16':        (['--batch-size', '16'], 'Larger batch size'),
    'no_focal':     (['--no-focal'], 'BCE+Dice loss instead of Focal'),
    'pos_weight_5': (['--no-focal', '--pos-weight', '5'], 'BCE+Dice, pos_weight=5'),
    'pos_weight_20':(['--no-focal', '--pos-weight', '20'], 'BCE+Dice, pos_weight=20'),
}
print(f'{len(ABLATIONS)} ablations configured')

## 3. Run Ablation Experiments

In [ ]:
train_script = str(Path.cwd().parent / 'scripts' / 'train.py')
results = []
for name, (extra_args, desc) in ABLATIONS.items():
    cmd = [sys.executable, train_script, '--epochs', '5', '--seed', '42',
           '--output-dir', str(OUT / name)] + extra_args
    print(f'\n{"-"*60}\n{name}: {desc}')
    start = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start
    best_dice, test_dice = None, None
    for line in (r.stdout + r.stderr).split('\n'):
        if 'Best val dice' in line:
            best_dice = float(line.split(':')[-1].strip())
        if 'Test Dice' in line:
            test_dice = float(line.split(':')[1].split(',')[0].strip())
    results.append({'name': name, 'description': desc, 'best_val_dice': best_dice,
                    'test_dice': test_dice, 'time_min': elapsed/60, 'exit_code': r.returncode})
    print(f'  val_dice={best_dice}, test_dice={test_dice}, time={elapsed/60:.1f}min')

## 4. Results Table

In [ ]:
df = pd.DataFrame(results)
df = df.sort_values('best_val_dice', ascending=False).reset_index(drop=True)
print(df[['name', 'best_val_dice', 'test_dice', 'time_min']].to_string())
print(f'\nBest config: {df.iloc[0]["name"]} (val_dice={df.iloc[0]["best_val_dice"]:.4f})')

## 5. Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['green' if r['best_val_dice'] == df['best_val_dice'].max() else 'steelblue' for r in results]
ax.barh(range(len(df)), df['best_val_dice'], color=colors, alpha=0.8)
ax.set_yticks(range(len(df))); ax.set_yticklabels(df['name'])
ax.set_xlabel('Best Validation Dice'); ax.set_title('Ablation Study Results')
ax.grid(True, axis='x', alpha=0.3)
for i, v in enumerate(df['best_val_dice']):
    if v is not None: ax.text(v+0.005, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUT / 'ablation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Results

In [ ]:
summary = {
    'results': results,
    'best_config': df.iloc[0]['name'],
    'best_val_dice': float(df.iloc[0]['best_val_dice']),
}
with open(OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved to {OUT / "summary.json"}')

print('\nKey findings:')
print(f'  FocalLoss vs BCE+Dice: compare baseline vs no_focal')
print(f'  Pretraining impact: compare baseline vs no_pretrain')
print(f'  Learning rate sensitivity: compare lr_1e-4, baseline, lr_1e-2')